In [ ]:
# ============================================================================
# MASTER MLflow LOGGING - DAGSHUB INTEGRATION 
# Models and essential artifacts
# ============================================================================

import mlflow
import pickle
import pandas as pd
import os
from datetime import datetime
import json
from scipy.sparse import load_npz, save_npz
import tempfile
from dotenv import load_dotenv
import getpass
from mlflow.tracking import MlflowClient
import requests

print("=" * 60)
print("MASTER MLflow LOGGING - DAGSHUB INTEGRATION")
print("=" * 60)

# ============================================================================
# DAGSHUB CONFIGURATION
# ============================================================================

print("\n" + "-" * 40)
print("DAGSHUB CREDENTIALS SETUP")
print("-" * 40)

# Load environment variables
load_dotenv()

# Get credentials from environment or prompt
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD") 
REPO_NAME = os.getenv("MLFLOW_REPO_NAME")

# If not in env, prompt user
if not MLFLOW_TRACKING_USERNAME:
    MLFLOW_TRACKING_USERNAME = input("Enter your DagsHub username: ").strip()
if not MLFLOW_TRACKING_PASSWORD:
    MLFLOW_TRACKING_PASSWORD = getpass.getpass("Enter your DagsHub token: ").strip()
if not REPO_NAME:
    REPO_NAME = input("Enter your FULL DagsHub repo (username/repo_name): ").strip()

# Set credentials in environment
os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD

# Clean repo name and construct URI
CLEAN_REPO = REPO_NAME.strip("/")
DAGSHUB_TRACKING_URI = f"https://dagshub.com/{CLEAN_REPO}.mlflow"
mlflow.set_tracking_uri(DAGSHUB_TRACKING_URI)

print(f"✅ Authenticated as: {MLFLOW_TRACKING_USERNAME}")
print(f"✅ Tracking URI: {mlflow.get_tracking_uri()}")

# ============================================================================
# CONNECTION PRE-FLIGHT CHECK
# ============================================================================

print("\n" + "-" * 40)
print("🔍 TESTING DAGSHUB CONNECTION")
print("-" * 40)

try:
    # Test the connection with a simple API call
    client = MlflowClient()
    # Try to search experiments (lightweight call)
    experiments = client.search_experiments(max_results=1)
    print("✅ Successfully connected to DagsHub API!")
    print(f"   Found {len(experiments)} existing experiments")
except Exception as e:
    print(f"❌ Failed to connect to DagsHub.")
    print(f"   Error: {str(e)}")
    print(f"\n   Troubleshooting:")
    print(f"   1. Verify your token has correct permissions")
    print(f"   2. Check that repo '{CLEAN_REPO}' exists")
    print(f"   3. Try accessing: https://dagshub.com/{CLEAN_REPO}")
    print(f"\n   Stopping execution to prevent further errors.")
    raise e

# ============================================================================
# CREATE/GET EXPERIMENT
# ============================================================================

EXPERIMENT_NAME = "Movie_Recommender_Master"
try:
    # Try to get existing experiment first
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment:
        experiment_id = experiment.experiment_id
        print(f"✅ Using existing experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
    else:
        experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
        print(f"✅ Created new experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
except Exception as e:
    print(f"⚠️ Experiment setup warning: {e}")
    mlflow.set_experiment(EXPERIMENT_NAME)
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    print(f"✅ Using fallback: {EXPERIMENT_NAME} (ID: {experiment.experiment_id})")

# ============================================================================
# FUNCTION TO LOG ANY FILE TO MLflow
# ============================================================================

def log_file_to_mlflow(file_path, artifact_path):
    """Log any file to MLflow with metadata"""
    if not os.path.exists(file_path):
        print(f"  ⚠ File not found: {file_path}")
        return False
    
    file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
    mlflow.log_artifact(file_path, artifact_path)
    
    # Log file metadata as a metric
    metric_name = f"{artifact_path.replace('/', '_')}_size_mb"
    mlflow.log_metric(metric_name, file_size)
    
    print(f"  ✓ Logged: {os.path.basename(file_path)} ({file_size:.2f} MB) → {artifact_path}")
    return True

# ============================================================================
# START MASTER RUN
# ============================================================================

run_name = f"master_log_{datetime.now().strftime('%Y%m%d_%H%M')}"
with mlflow.start_run(run_name=run_name) as master_run:
    
    run_id = master_run.info.run_id
    run_url = f"https://dagshub.com/{CLEAN_REPO}/experiments/#/experiments/{master_run.info.experiment_id}/runs/{run_id}"
    
    print(f"\n✅ Master Run Started:")
    print(f"  Run ID: {run_id}")
    print(f"  Run Name: {run_name}")
    print(f"  DagsHub URL: {run_url}")
    
    # ------------------------------------------------------------------------
    # LOG PARAMETERS - SYSTEM INFO
    # ------------------------------------------------------------------------
    
    mlflow.log_param("system", "Local Machine")
    mlflow.log_param("timestamp", datetime.now().isoformat())
    mlflow.log_param("python_version", os.sys.version)
    mlflow.log_param("dagshub_user", MLFLOW_TRACKING_USERNAME)
    mlflow.log_param("dagshub_repo", CLEAN_REPO)
    
    # ------------------------------------------------------------------------
    # LOG ALL SPARSE MATRICES (NPZ files)
    # ------------------------------------------------------------------------
    
    print("\n" + "-" * 40)
    print("📁 LOGGING SPARSE MATRICES")
    print("-" * 40)
    
    npz_files = [
        ("../data/processed/als_confidence_matrix.npz", "matrices/als_confidence"),
        ("../data/processed/als_training_matrix.T.npz", "matrices/als_training"),
        ("../data/processed/interactions_matrix.npz", "matrices/interactions"),
    ]
    
    npz_count = 0
    for file_path, artifact_path in npz_files:
        if log_file_to_mlflow(file_path, artifact_path):
            npz_count += 1
    
    # ------------------------------------------------------------------------
    # LOG ALL CBF MODEL ARTIFACTS
    # ------------------------------------------------------------------------
    
    print("\n" + "-" * 40)
    print("🤖 LOGGING CBF MODEL ARTIFACTS")
    print("-" * 40)
    
    cbf_files = [
        ("../models/cbf/tfidf_vectorizer.pkl", "models/cbf/tfidf_vectorizer"),
        ("../models/cbf/svd_model.pkl", "models/cbf/svd_model"),
        ("../models/cbf/faiss_index.bin", "models/cbf/faiss_index"),
        ("../models/cbf/movie_id_to_idx.pkl", "models/cbf/movie_id_to_idx"),
        ("../models/cbf/idx_to_movie_id.pkl", "models/cbf/idx_to_movie_id"),
        ("../models/cbf/genre_vectors.pkl", "models/cbf/genre_vectors"),
        ("../models/cbf/movies_df.pkl", "models/cbf/movies_df"),
    ]
    
    cbf_count = 0
    for file_path, artifact_path in cbf_files:
        if log_file_to_mlflow(file_path, artifact_path):
            cbf_count += 1
    
    # ------------------------------------------------------------------------
    # LOG ALL CF MODEL ARTIFACTS (ALS)
    # ------------------------------------------------------------------------
    
    print("\n" + "-" * 40)
    print("🤖 LOGGING CF MODEL ARTIFACTS")
    print("-" * 40)
    
    cf_files = [
        ("../models/cf/als_model.pkl", "models/cf/als_model"),
        ("../models/cf/als_model_eval.pkl", "models/cf/als_model_eval"),
        ("../models/cf/user_mapper.pkl", "models/cf/user_mapper"),
        ("../models/cf/user_inv_mapper.pkl", "models/cf/user_inv_mapper"),
        ("../models/cf/movie_mapper.pkl", "models/cf/movie_mapper"),
        ("../models/cf/movie_inv_mapper.pkl", "models/cf/movie_inv_mapper"),
        ("../models/cf/cf_inference_config.pkl", "models/cf/inference_config"),
    ]
    
    cf_count = 0
    for file_path, artifact_path in cf_files:
        if log_file_to_mlflow(file_path, artifact_path):
            cf_count += 1
    
    # ------------------------------------------------------------------------
    # LOG SENTIMENT MODEL ARTIFACTS
    # ------------------------------------------------------------------------
    
    print("\n" + "-" * 40)
    print("🤖 LOGGING SENTIMENT MODEL ARTIFACTS")
    print("-" * 40)
    
    sentiment_files = [
        ("../models/sentiment/sentiment_vectorizer.pkl", "models/sentiment/vectorizer"),
        ("../models/sentiment/sentiment_classifier.pkl", "models/sentiment/classifier"),
    ]
    
    sentiment_count = 0
    for file_path, artifact_path in sentiment_files:
        if log_file_to_mlflow(file_path, artifact_path):
            sentiment_count += 1
    
    # ------------------------------------------------------------------------
    # LOG NOTEBOOKS AS ARTIFACTS
    # ------------------------------------------------------------------------
    
    print("\n" + "-" * 40)
    print("📓 LOGGING NOTEBOOKS")
    print("-" * 40)
    
    notebook_files = [
        ("../notebooks/01_data_ingestion_eda.ipynb", "notebooks/01_eda"),
        ("../notebooks/02_feature_engineering_cbf.ipynb", "notebooks/02_feature_cbf"),
        ("../notebooks/03_cbf_model_training.ipynb", "notebooks/03_cbf_training"),
        ("../notebooks/04_feature_engineering_cf.ipynb", "notebooks/04_feature_cf"),
        ("../notebooks/05_cf_model_training.ipynb", "notebooks/05_cf_training"),
        ("../notebooks/06_hybrid_fusion.ipynb", "notebooks/06_hybrid"),
        ("../notebooks/07_evaluation.ipynb", "notebooks/07_evaluation"),
        ("../notebooks/08_sentiments.ipynb", "notebooks/08_sentiments"),
        ("../notebooks/09_mlflow_master_logging.ipynb", "notebooks/09_mlflow_master_logging"),
    ]
    
    notebook_count = 0
    for file_path, artifact_path in notebook_files:
        if log_file_to_mlflow(file_path, artifact_path):
            notebook_count += 1
    
    # ------------------------------------------------------------------------
    # CREATE AND LOG A MASTER INVENTORY FILE
    # ------------------------------------------------------------------------
    
    print("\n" + "-" * 40)
    print("📊 CREATING MASTER INVENTORY")
    print("-" * 40)
    
    inventory = {
        "run_id": run_id,
        "run_url": run_url,
        "timestamp": datetime.now().isoformat(),
        "dagshub": {
            "user": MLFLOW_TRACKING_USERNAME,
            "repo": CLEAN_REPO,
            "tracking_uri": DAGSHUB_TRACKING_URI
        },
        "models": {
            "cbf": {
                "files": [os.path.basename(f[0]) for f in cbf_files if os.path.exists(f[0])],
                "count": cbf_count,
                "logged": cbf_count
            },
            "cf": {
                "files": [os.path.basename(f[0]) for f in cf_files if os.path.exists(f[0])],
                "count": cf_count,
                "logged": cf_count
            },
            "sentiment": {
                "files": [os.path.basename(f[0]) for f in sentiment_files if os.path.exists(f[0])],
                "count": sentiment_count,
                "logged": sentiment_count
            }
        },
        "matrices": {
            "files": [os.path.basename(f[0]) for f in npz_files if os.path.exists(f[0])],
            "count": npz_count
        },
        "notebooks": {
            "files": [os.path.basename(f[0]) for f in notebook_files if os.path.exists(f[0])],
            "count": notebook_count
        }
    }
    
    # Save inventory to temp file and log
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        json.dump(inventory, f, indent=2)
        temp_path = f.name
    
    mlflow.log_artifact(temp_path, "inventory")
    os.unlink(temp_path)
    
    total_models = cbf_count + cf_count + sentiment_count
    print(f"✅ Master inventory logged with {total_models} total models")
    
    # ------------------------------------------------------------------------
    # LOG SUMMARY METRICS
    # ------------------------------------------------------------------------
    
    mlflow.log_metric("total_models", total_models)
    mlflow.log_metric("total_matrices", npz_count)
    mlflow.log_metric("total_notebooks", notebook_count)
    
    # ------------------------------------------------------------------------
    # TAGS
    # ------------------------------------------------------------------------
    
    mlflow.set_tag("project", "Movie Recommender System")
    mlflow.set_tag("status", "complete")
    mlflow.set_tag("cbf_trained", "true")
    mlflow.set_tag("cf_trained", "true")
    mlflow.set_tag("sentiment_trained", "true")
    mlflow.set_tag("storage", "DagsHub")
    mlflow.set_tag("environment", "Local")
    mlflow.set_tag("dagshub_repo", CLEAN_REPO)
    mlflow.set_tag("master_run", "true")
    mlflow.set_tag("csv_files_excluded", "true")

print("\n" + "=" * 60)
print("✅ MASTER MLflow LOGGING COMPLETE - DAGSHUB")
print("=" * 60)
print(f"\n📊 Your MLflow Dashboard:")
print(f"   https://dagshub.com/{CLEAN_REPO}/experiments/")
print(f"\n🔍 This Run URL:")
print(f"   {run_url}")
print(f"\n📦 Summary:")
print(f"   - Models logged: {total_models}")
print(f"   - Matrices: {npz_count}")
print(f"   - Notebooks: {notebook_count}")
print("\n✅ Essential artifacts permanently stored on DagsHub")

MASTER MLflow LOGGING - DAGSHUB INTEGRATION

----------------------------------------
DAGSHUB CREDENTIALS SETUP
----------------------------------------
✅ Authenticated as: ayanfeoluwadegoke
✅ Tracking URI: https://dagshub.com/ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System.mlflow

----------------------------------------
🔍 TESTING DAGSHUB CONNECTION
----------------------------------------
✅ Successfully connected to DagsHub API!
   Found 1 existing experiments
✅ Using existing experiment: Movie_Recommender_Master (ID: 1)

✅ Master Run Started:
  Run ID: 17570ac64fc54303a358a15d4bd9fac0
  Run Name: master_log_20260301_1744
  DagsHub URL: https://dagshub.com/ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System/experiments/#/experiments/0/runs/17570ac64fc54303a358a15d4bd9fac0

----------------------------------------
📁 LOGGING SPARSE MATRICES
----------------------------------------
  ✓ Logged: als_confidence_matrix.npz (58.43 MB) → matrices/als_confidence
  ✓ Logged: als_training_